# Questão 2
## 1. Motivação


## 2. Hipótese


In [22]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [23]:
df = pd.read_parquet("../data/resultados_candidato.parquet")
df.shape

(2925021, 12)

In [24]:
df["eleito"] = df["resultado"].fillna("").str.startswith("eleito")

# presidente: o resultado falta em 2006 e 2010; usa-se o mais votado do último turno
pres = df[(df["cargo"] == "presidente") & (df["tipo_eleicao"] == "eleicao ordinaria") & df["turno"].notna()].copy()
pres = pres[pres["turno"] == pres.groupby("ano")["turno"].transform("max")]
venc = pres.loc[pres.groupby("ano")["votos"].idxmax()]
df.loc[venc.index, "eleito"] = True
print(venc[["ano", "sigla_partido", "genero", "votos", "resultado"]])

# contagem de eleitos nas eleições ordinárias
ordin = df[df["tipo_eleicao"] == "eleicao ordinaria"]
alvo = ["deputado federal", "senador", "governador", "presidente"]
ordin[ordin["eleito"] & ordin["cargo"].isin(alvo)].groupby(["ano", "cargo"]).size().unstack("cargo")

          ano sigla_partido     genero     votos resultado
819699   2002            PT  masculino  52793364    eleito
664722   2006            PT  masculino  58295042      None
1838514  2010            PT   feminino  55752529      None
870989   2014            PT   feminino  54501118    eleito
1188561  2018           PSL  masculino  57797847    eleito
311571   2022            PT  masculino  60345999    eleito


cargo,deputado federal,governador,presidente,senador
ano,,,,
2002,513,27,1,54
2006,513,27,1,27
2010,513,27,1,54
2014,513,27,1,27
2018,513,27,1,53
2022,513,27,1,27


In [25]:
cand = pd.read_parquet("../data/candidatos.parquet", columns=["ano"])
print(len(df), cand["ano"].between(2002, 2024).sum())

2925021 2925021


In [26]:
n08 = df[(df["ano"] == 2008) & df["cargo"].isin(["vereador", "prefeito"]) & df["resultado"].isna() & df["turno"].notna()]
print(len(n08))
print(n08["genero"].value_counts(normalize=True))

465
genero
masculino    0.797849
feminino     0.202151
Name: proportion, dtype: float64


In [29]:
elei08 = df[(df["ano"] == 2008) & df["cargo"].isin(["vereador", "prefeito"]) & df["eleito"]]
print(len(n08) / len(elei08) * 100)   # nulos como % dos eleitos de 2008

0.810005748427892


## Análise

In [31]:
base = df[df["tipo_eleicao"] == "eleicao ordinaria"].copy()
base = base[base["genero"].isin(["feminino", "masculino"])]

proporcionais = ["vereador", "deputado estadual", "deputado federal", "deputado distrital"]
majoritarios  = ["prefeito", "governador", "senador", "presidente"]
base = base[base["cargo"].isin(proporcionais + majoritarios)]

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas)

In [35]:
def pct_fem(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).size().unstack("genero", fill_value=0)
    t["total"] = t["feminino"] + t["masculino"]
    t["pct_fem"] = t["feminino"] / t["total"] * 100
    return t.reset_index()

cand = pct_fem(base, ["ano", "cargo"])
eleit = pct_fem(base[base["eleito"]], ["ano", "cargo"])

cmp_ = cand.merge(eleit, on=["ano", "cargo"], suffixes=("_cand", "_eleit"))
cmp_["dif_pp"] = cmp_["pct_fem_eleit"] - cmp_["pct_fem_cand"]

In [38]:
v = base[base["valida"]]
t = v.groupby(["ano", "cargo", "genero"], observed=True)["eleito"].agg(["sum", "count"])
t["taxa"] = t["sum"] / t["count"] * 100
taxa = t["taxa"].unstack("genero")
taxa["razao_f_m"] = taxa["feminino"] / taxa["masculino"]

In [39]:
v = base[base["valida"]]

t = v.groupby(["ano", "cargo", "genero"], observed=True)["eleito"].agg(["sum", "count"])
t["taxa"] = t["sum"] / t["count"] * 100

taxa = t["taxa"].unstack("genero")
taxa["razao_f_m"] = taxa["feminino"] / taxa["masculino"]
taxa = taxa.reset_index()

In [40]:
taxa[taxa["cargo"].isin(proporcionais)].round({"feminino": 1, "masculino": 1, "razao_f_m": 2})

genero,ano,cargo,feminino,masculino,razao_f_m
0,2002,deputado distrital,3.9,3.8,1.02
1,2002,deputado estadual,8.0,9.5,0.85
2,2002,deputado federal,8.7,12.7,0.69
7,2004,vereador,8.5,16.8,0.51
8,2006,deputado distrital,2.2,4.1,0.54
9,2006,deputado estadual,7.6,9.3,0.82
10,2006,deputado federal,7.2,10.8,0.66
15,2008,vereador,9.1,17.9,0.51
16,2010,deputado distrital,2.0,3.4,0.59
17,2010,deputado estadual,5.5,9.6,0.57


In [36]:
cmp_[cmp_["cargo"].isin(proporcionais)].round(1)

genero,ano,cargo,feminino_cand,masculino_cand,total_cand,pct_fem_cand,feminino_eleit,masculino_eleit,total_eleit,pct_fem_eleit,dif_pp
0,2002,deputado distrital,137,526,663,20.7,5,19,24,20.8,0.2
1,2002,deputado estadual,1686,9871,11557,14.6,129,906,1035,12.5,-2.1
2,2002,deputado federal,501,3831,4332,11.6,42,471,513,8.2,-3.4
7,2004,vereador,81074,286834,367908,22.0,6537,45186,51723,12.6,-9.4
8,2006,deputado distrital,134,511,645,20.8,3,21,24,12.5,-8.3
9,2006,deputado estadual,1722,10474,12196,14.1,121,914,1035,11.7,-2.4
10,2006,deputado federal,665,4586,5251,12.7,45,468,513,8.8,-3.9
15,2008,vereador,76846,272406,349252,22.0,6487,45290,51777,12.5,-9.5
16,2010,deputado distrital,224,660,884,25.3,4,20,24,16.7,-8.7
17,2010,deputado estadual,3246,11019,14265,22.8,134,901,1035,12.9,-9.8


## Decisões

Só eleições ordinárias, sem as suplementares.
“Eleito” inclui as três formas (eleito, eleito por qp, eleito por media). Suplentes não contam.
Presidente em 2006 e 2010: o eleito foi identificado pelo maior número de votos no último turno, porque o resultado está ausente na base.
Em 2008, uma parte mínima das candidaturas (cerca de 0,81% dos eleitos) não tem resultado registrado e foi tratada como não eleita.

In [27]:
sen = df[(df["cargo"] == "senador") & (df["ano"] == 2018)]
print(sen["tipo_eleicao"].value_counts())

por_uf = sen[sen["eleito"]].groupby(["tipo_eleicao", "sigla_uf"]).size()
print(por_uf[por_uf != 2])          # estados com número de eleitos diferente de 2

tipo_eleicao
eleicao ordinaria                   356
senador                              13
eleicao suplementar senador - mt     11
Name: count, dtype: int64
tipo_eleicao                      sigla_uf
eleicao ordinaria                 MT          1
eleicao suplementar senador - mt  MT          1
dtype: int64


In [28]:
uf = "MT"
sen[sen["sigla_uf"] == uf].sort_values("votos", ascending=False)[
    ["nome", "genero", "situacao", "resultado", "votos"]].head(6) if "nome" in sen else \
sen[sen["sigla_uf"] == uf].sort_values("votos", ascending=False)[
    ["genero", "situacao", "resultado", "votos"]].head(6)

,genero,situacao,resultado,votos
2366439,masculino,deferido,eleito,490699
2346877,masculino,deferido,nao eleito,434972
2322587,masculino,deferido,eleito,371857
1864763,masculino,deferido,nao eleito,333082
1495810,masculino,deferido,nao eleito,330430
481348,feminino,deferido,nao eleito,293362


## Limitações
Em 2018, uma das vagas de senador por Mato Grosso consta na base como eleição suplementar e, por isso, não entra na análise (53 eleitos em vez de 54). O candidato eleito nessa vaga era homem, de modo que a exclusão não afeta a contagem de mulheres eleitas.